In [1]:
import re
import warnings
import numpy as np
import pandas as pd

from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import ComplementNB
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import f1_score

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
N_SPLITS = 5

#### Определяем функции для очистки текста и объединения полей title + body в один текстовый признак

In [2]:
def preprocess(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def build_text(df):
    return (df["title"].fillna("") + " " + df["body"].fillna("")).apply(preprocess)

#### Загружаем обучающую и тестовую выборки, формируем текстовые признаки и целевую переменную

In [3]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

X_train_text = build_text(train)
X_test_text = build_text(test)
y_train = train["label"].astype(int)

#### Создаём TF-IDF признаки: отдельно по словам и по символьным n-граммам

In [4]:
word_vectorizer = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.9,
    sublinear_tf=True
)

char_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    max_features=50000,
    min_df=2,
    sublinear_tf=True
)

#### Обучаем векторизаторы на обучающей выборке и применяем к train и test, объединяем словарные и символьные признаки в одну матрицу

In [5]:
X_train_word = word_vectorizer.fit_transform(X_train_text)
X_test_word = word_vectorizer.transform(X_test_text)

X_train_char = char_vectorizer.fit_transform(X_train_text)
X_test_char = char_vectorizer.transform(X_test_text)

X_train_final = hstack([X_train_word, X_train_char])
X_test_final = hstack([X_test_word, X_test_char])

#### Набор моделей: логистическая регрессия, наивный Байес и SGD-классификатор (подобраны методом проб и ошибок - перебором)

In [6]:
models = {
    "lr": LogisticRegression(
        max_iter=2000,
        C=4.0,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "nb": ComplementNB(alpha=0.3),
    "sgd": SGDClassifier(
        loss="log_loss",
        alpha=1e-5,
        penalty="l2",
        class_weight="balanced",
        random_state=RANDOM_STATE,
        max_iter=1000,
        tol=1e-3
    )
}

#### Обучаем каждую модель с кросс-валидацией, получаем OOF-предсказания и считаем F1

In [8]:
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

oof_preds = {}
test_preds = {}
scores = {}

for name, model in models.items():
    print(f"Training {name.upper()}")

    oof_proba = cross_val_predict(
        model,
        X_train_final,
        y_train,
        cv=cv,
        method="predict_proba",
        n_jobs=-1
    )[:, 1]

    oof_preds[name] = oof_proba
    score = f1_score(y_train, (oof_proba >= 0.5).astype(int))
    scores[name] = score
    print(f"{name.upper()} OOF F1: {score:.4f}")

    model.fit(X_train_final, y_train)
    test_preds[name] = model.predict_proba(X_test_final)[:, 1]

Training LR
LR OOF F1: 0.8219
Training NB
NB OOF F1: 0.7797
Training SGD
SGD OOF F1: 0.8169


#### Формируем ансамбль моделей, используя веса пропорционально их качеству F1

In [9]:
weights = np.array(list(scores.values()), dtype=float)
weights = weights / weights.sum()

ensemble_proba = np.zeros(X_test_final.shape[0], dtype=float)
for i, name in enumerate(models.keys()):
    ensemble_proba += weights[i] * test_preds[name]

oof_ensemble = np.zeros(X_train_final.shape[0], dtype=float)
for i, name in enumerate(models.keys()):
    oof_ensemble += weights[i] * oof_preds[name]

#### Подбираем оптимальный порог классификации для максимизации F1 на OOF-предсказаниях

In [10]:
thresholds = np.arange(0.2, 0.81, 0.01)
best_thr, best_f1 = 0.5, 0.0

for thr in thresholds:
    f1 = f1_score(y_train, (oof_ensemble >= thr).astype(int))
    if f1 > best_f1:
        best_f1 = f1
        best_thr = thr

print(f"Best threshold: {best_thr:.2f}")
print(f"OOF Ensemble F1: {best_f1:.4f}")

Best threshold: 0.53
OOF Ensemble F1: 0.8345


#### Финальные предсказания

In [12]:
final_pred = (ensemble_proba >= best_thr).astype(int)

submission = pd.DataFrame({
    "id": test["id"],
    "label": final_pred
})

submission.to_csv("submission.csv", index=False)

print("Submission file created")
print(f"Predicted {final_pred.sum()} positive cases out of {len(final_pred)} total")

Submission file created
Predicted 231 positive cases out of 973 total
